# Harvest DataCite dataset records up to September 2025

## Import

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sindex.sources.datacite.jobs import (
    harvest_datacite_datasets_for_date_range_to_ndjson,
    batch_slim_datacite_record_to_ndjson,
    batch_slim_datacite_record_to_ndjson_fast,
    batch_slim_datacite_chunked #FASTEST
    )
from sindex.utils.datasets import create_datasets_db_from_ndjson
from pathlib import Path
import os
import duckdb

## Query datacite and write to ndjson files

In [14]:
#### Test parameters
#start_date_str = "2014-12-28"
#end_date_str = "2015-01-10"
#out_dir = Path("output/datacite/harvest_date_range_test")

# Actual
start_date_str = "2011-03-08"
end_date_str = "2016-01-16" #update this date if code fails to the one that was running
out_dir = Path("C:/Users/BPatel/Documents/batch-data/datacite-raw")

out_dir.mkdir(parents=True, exist_ok=True)

n = harvest_datacite_datasets_for_date_range_to_ndjson(
    start_date_str=start_date_str,
    end_date_str=end_date_str,
    window_days=7,
    page_size=1000,
    save_folder=str(out_dir),
    skip_empty_files=True,
    polite_sleep_seconds=0.5,
)

print(f"Done. Wrote {n:,} records across window files in {out_dir.resolve()}")

Fetching records 2016-01-10 → 2016-01-16 (window_days=7, page_size=1000, detail=True)
  Saved 11872 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2016-01-10-2016-01-16.ndjson
Fetching records 2016-01-03 → 2016-01-09 (window_days=7, page_size=1000, detail=True)
  Saved 5750 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2016-01-03-2016-01-09.ndjson
Fetching records 2016-01-01 → 2016-01-02 (window_days=7, page_size=1000, detail=True)
  Saved 206 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2016-01-01-2016-01-02.ndjson
Fetching records 2015-12-25 → 2015-12-31 (window_days=7, page_size=1000, detail=True)
  Saved 2484 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2015-12-25-2015-12-31.ndjson
Fetching records 2015-12-18 → 2015-12-24 (window_days=7, page_size=1000, detail=True)
  Saved 4182 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2015-12-18-2015-12-24.ndjson
Fetching record

---------------
---------------
Check results on DataCite commons: https://commons.datacite.org/doi.org?query=%28types.resourceTypeGeneral%3ADataset%29+AND+%28created%3A%5B2025-09-24+TO+2025-09-30%5D%29&registration-agency=datacite

## Check number of records

In [7]:
# In one file
def count_ndjson_lines(file_path):
    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  
                count += 1
    return count
file_path = os.path.join(out_dir, "datacite-2025-07-30-2025-08-05.ndjson")
count = count_ndjson_lines(file_path)
print(f"Total lines: {count}")

Total lines: 1394000


In [ ]:
# Across all files

## Write batch slim records

In [4]:
src_folder = r"D:\pipeline-data\records\raw-records\datacite-records"
#src_folder = r"D:\pipeline-data\records\other\demo_raw_record"
dst_folder =r"D:\pipeline-data\records\slim-records\datacite-slim-records"

In [5]:
summary = batch_slim_datacite_chunked(
    src_folder=src_folder,
    dst_folder=dst_folder)

Found 771 input files.
Starting processing with 32 cores. Batch size: 100,000...
Processed: 49,009,522 lines | Batches: 491 | Bad: 0
--------------------------------------------------
DONE in 4709.05s
Total Lines Read: 49,009,522
Total Lines Kept: 49,009,522
Processing Rate:  10,407 records/sec
Output Files:     491 files written to D:\pipeline-data\records\slim-records\datacite-slim-records
--------------------------------------------------


## Slim records to duckdb

### Test

In [17]:
slim_folder = r"C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\input\demo_datasets_slim_metadata_ndjson"
db_path = r"C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks-batch\output\datacite\test_datacite_slim_records.duckdb"
create_datasets_db_from_ndjson(slim_folder, db_path)

1/3: Extracting data from 2 files...
2/3: Creating index for fast lookups (this may take a moment)...
3/3: Saving to disk...

Success! Persistent DB created at: C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks-batch\output\datacite\test_datacite_slim_records.duckdb
Total indexed datasets: 3


In [18]:
con = duckdb.connect(db_path)
row_count = con.execute("SELECT count() FROM my_datasets").fetchone()[0]
print(f"Total datasets in table: {row_count}")
display(con.execute("SELECT * FROM my_datasets LIMIT 5").df())
con.close()

Total datasets in table: 3


,dataset_id,id_type,publication_date,created_date,publication_year
0,10.13026/kpb9-mt58,doi,2024-10-10T21:27:19+00:00,None,2024
1,10.60775/fairhub.2,doi,2024-10-28T17:30:18+00:00,None,2024
2,EMD-24511,emdb_id,2021-07-22T00:00:00,None,2021


### Actual

In [6]:
slim_folder = r"D:\pipeline-data\records\slim-records\datacite-slim-records"
db_path = r"D:\pipeline-data\records\slim-records\datacite-slim-records.duckdb"

In [7]:
create_datasets_db_from_ndjson(slim_folder, db_path)

Extracting data from 491 files


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creating index for fast lookups (this may take a moment)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saving to disk

Success! Persistent DB created at: D:\pipeline-data\records\slim-records\datacite-slim-records.duckdb
Total indexed datasets: 49,009,522


In [8]:
con = duckdb.connect(db_path)
row_count = con.execute("SELECT count() FROM my_datasets").fetchone()[0]
print(f"Total datasets in table: {row_count:,}")
display(con.execute("SELECT * FROM my_datasets LIMIT 5").df())
con.close()

Total datasets in table: 49,009,522


,dataset_id,id_type,pubyear
0,10.5284/1000389,doi,2011
1,10.5284/1000140,doi,2011
2,10.5284/1000146,doi,2011
3,10.5284/1000144,doi,2011
4,10.5284/1000181,doi,2011


In [7]:
# Export to CSV
con = duckdb.connect(db_path)

copy_query = """
COPY (
    SELECT dataset_id 
    FROM my_datasets 
    WHERE dataset_id IS NOT NULL AND dataset_id != ''
) TO 'exported_dataset_ids.csv' (HEADER);
"""

# Execute the command
con.execute(copy_query)
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Number of data repositories/publishers

In [5]:
input_pattern = r'D:\pipeline-data\records\slim-records\datacite-slim-records\*.ndjson'
output_file = r'D:\pipeline-data\records\publisher_counts.csv'

In [8]:
import json
import glob
import csv
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_file(file_path):
    local_counts = Counter()
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: 
                    continue
                try:
                    data = json.loads(line)
                    raw_publisher = data.get('publisher')
                    
                    if raw_publisher:
                        pub = str(data.get('publisher') or 'unknown').strip().lower()
                        local_counts[pub] += 1
                    else:
                        local_counts['Unknown'] += 1
                        
                except json.JSONDecodeError:
                    continue
    except Exception as e:
        print(f"\nSkipping file {file_path}: {e}")
    return local_counts


files = glob.glob(input_pattern)
global_counts = Counter()
processed_count = 0

print(f"Processing {len(files)} files...")

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_file = {executor.submit(process_single_file, f): f for f in files}
    
    for future in as_completed(future_to_file):
        result_counter = future.result()
        global_counts.update(result_counter)
        processed_count += 1
        
        if processed_count % 10 == 0:
            sys.stdout.write(f"\rFiles: {processed_count}/{len(files)}")
            sys.stdout.flush()

with open(output_file, 'w', newline='', encoding='utf-8-sig') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Publisher', 'Dataset Count'])
    for pub, count in global_counts.most_common():
        writer.writerow([pub, count])

print(f"\nDone! Saved to {output_file} .")
print(f"\n\nSuccess! Final unique count: {len(global_counts)}")

Processing 491 files...
Files: 490/491
Done! Saved to D:\pipeline-data\records\publisher_counts.csv .


Success! Final unique count: 17306
